In [ ]:
import polars as pl
%pip install altair
%pip install fastexcel
import altair
import regex


In [ ]:
"""
Exploration conclusions:
    1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings
        
        Example:
        
        [contract_desc]                     [nip]
        "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"
        
        Decisions:
        
        There are 14 rows where after splitting data, lengths in columns don't match. These will be removed from further analysis. 
        2. Column anomalies:
        * year_filename - no anomalies (one ghost row containing nulls for all columns, other than that fits withing the predicted range, and is i64)
        * gross_total_pln - the maximum amount found is 3.4434e9 which most likely means that during the conversion from excel, the numbers got corrupted
        * date_valid_from & date_valid_to 
            - there are dates that make the contract stretch past the period expected to be included + contracts with no completion date
            - no null values for date_valid_from (except a ghost row with all nulls)
            - 282 null values for date_valid_to
            - no dates that make the whole period negative 
            - lowest contract duration == 0
"""


In [ ]:
df = pl.read_excel(r"data\umk_contracts_202*.xlsx")

df_org = df.rename({
    'Lp': 'idx',
    'rok_plik': 'year_filename',
    'Numer w rejestrze': 'registry',
    'Nazwa Kontrahenta ': 'contractor_name',
    'NIP': 'nip',
    'Kwota złotych brutto': 'gross_total_pln',
    'Data obowiązywania od': 'date_valid_from',
    'Data obowiązywania do': 'date_valid_to',
    'Data zawarcia': 'date_contract_signed',
    'Przedmiot umowy': 'contract_desc',
    'Jedn. Realizująca': 'administrative_unit'
})

from types import SimpleNamespace

_names = SimpleNamespace({col: col for col in df_org.columns})

In [ ]:
"""
1. Cols [contract_desc] and [nip] are not atomic - data is paired in a form of comma delimited strings
    
    Example:
    
    [contract_desc]                     [nip]
    "company_a, company_b, company_c" | "6791921738, 8691984858, 78236424843"
    
    Decisions:
    
    There are 14 rows where after splitting data, lengths in columns don't match. These will be removed from further analysis. 
"""

df = (
    df_org
    .select(
        pl.col(_names.idx, _names.contractor_name, _names.nip)
    )
    .with_columns(
        pl.col(_names.nip).str.contains(',').alias("contains_comma")
    )
    # .select(
    #     pl.col("contains_comma").sum()
    # )
    .filter(
        pl.col("contains_comma")
    )
    .with_columns(
        pl.col(_names.contractor_name).str.split(','),
        pl.col(_names.nip).str.split(',')
    )
    .with_columns(
        abs(pl.col(_names.contractor_name).list.len().cast(pl.Int64) - pl.col(_names.nip).list.len().cast(pl.Int64)).alias("diff_len") #cast to Int64 or we get underflow
    )
    .filter(
        pl.col("diff_len") != 0
    )
)


In [ ]:
df_temp = df #[200:300]
df_temp

In [ ]:
pl.Config.set_fmt_str_lengths(300)
pl.Config.set_tbl_width_chars(100)

(
    df_temp
    .select(
    pl.col(_names.idx, _names.contractor_name, _names.nip)
    ).with_columns(pl.col(_names.nip).list.join(','),pl.col(_names.contractor_name).list.join(','))
    .with_columns(
        pl.col(_names.nip).str.contains('-').alias("contains_comma")
    )
    .filter(
        pl.col("contains_comma")
    )
    .with_columns(
         pl.col(_names.contractor_name).str.split(','),
         pl.col(_names.nip).str.split(',')
    )
    .with_columns(
         pl.col(_names.contractor_name).list.len().cast(pl.Int64).alias("contractor_name_len"),
         pl.col(_names.nip).list.len().cast(pl.Int64).alias("nip_len"),
         abs(pl.col(_names.contractor_name).list.len().cast(pl.Int64) - pl.col(_names.nip).list.len().cast(pl.Int64)).alias("diff_len") #cast to Int64 or we get underflow
    )
    .filter(
         (pl.col("diff_len") != 0)
         & (pl.col("contractor_name_len") > pl.col("nip_len"))
         & (pl.col("nip_len") == 2)
     )
)



In [ ]:
df_temp.filter(pl.col("idx") == 3865)

In [ ]:
"""
2. Column anomalies:
    * year_filename - no anomalies (one ghost row containing nulls for all columns, other than that fits withing the predicted range, and is i64)
    * gross_total_pln - the maximum amount found is 3.4434e9 which means that during the conversion from excel, the numbers got corrupted
    * date_valid_from & date_valid_to 
        - there are dates that make the contract stretch past the period expected to be included + contracts with no completion date
        - no null values for date_valid_from (except a ghost row with all nulls)
        - 282 null values for date_valid_to
        - no dates that make the whole period negative 
        - lowest contract duration == 0
    *registry 
        - registry structure: [TYPE] / [GRU_SECTION] / [SECTION_SEQ_NO] / [DEPARTMENT] / [DEPT_SEQ_NO] / [YEAR]
        or can be found on page 9 of the link below (except for the TYPE prefix which i cannot find)
        https://www.bing.com/ck/a?!&&p=5b2b128f642c5cc889b2953dbcacf3bbd4a526029baffb4c6fed4dd79636f5aaJmltdHM9MTc4NzYxNjAwMA&ptn=3&ver=2&hsh=4&fclid=36f48feb-4c30-6d89-25a6-98614d046cdb&psq=Zarz%c4%85dzenie+PMK+nr+559%2f2023+z+dnia+1+marca+2023+r&u=a1aHR0cHM6Ly93d3cuYmlwLmtyYWtvdy5wbC96YXJ6YWR6ZW5pYS8yMDIzLzU1OS9YMTlCUzE5Zk9EUXlOVE09LzU1OS0yMDIzLnBkZg
      all rows (except the ghost row) start with the prefix W which means that they refer to wydatki (at least thats my assumption for now) from UMK so it matches with what we are looking for
      the GRU_SECTION has also additional explanations that will be mapped to the df in the future:
        GRU_SECTION_MAP = {
        "I": "Current expenditure contracts",
        "II": "Investment contracts",
        "IV-B": "Orders financed from current expenditures",
        "IV-I": "Investment orders",
        "V": "Agreements",
        "VI": "Other contracts",
        "IX": "Loan contracts",
        "XI": "Contracts preceded by draft Mayor's order",
        "XII": "Lease/Rental contracts",
        "XIII": "Online purchase contracts",
        "XV": "Other budget liabilities",
    }


"""

In [ ]:
print(df_org.select(pl.col('year_filename').unique()))
#print(df_org.filter(pl.col('year_filename').is_null()))

In [ ]:
print(df_org.select(pl.col('gross_total_pln').is_null().unique())) #instead of just unique, used is_null for clarity as there is a lot of unique values for this column
print(df_org.select(pl.col('gross_total_pln').max()))

In [ ]:
#print(df_org.select(pl.col('date_valid_to')))
df_date_check = (
    df_org
    .with_columns(
        pl.col('date_valid_to').str.to_date('%Y-%m-%d'),
        pl.col('date_valid_from').str.to_date('%Y-%m-%d')
    )
    .with_columns(
        (pl.col('date_valid_to') - pl.col('date_valid_from')).dt.total_days().alias('duration_days')
    )
)
print(df_date_check.filter(pl.col('date_valid_from') ==df_date_check['date_valid_from'].min()))
print(df_date_check.filter(pl.col('date_valid_to') == df_date_check['date_valid_to'].max()))
#print(df_date_check.filter(pl.col('duration_days') == df_date_check['duration_days'].min()))

In [ ]:
#df_date_check.filter(pl.col('date_valid_from').dt.year() != pl.col('year_filename')) 
df_date_check.filter((pl.col('date_valid_from').dt.year() != pl.col('year_filename')) & pl.col('date_valid_to').is_null()) 

In [ ]:
print(df_date_check.select(pl.col('date_valid_from').null_count()))
print(df_date_check.filter(pl.col('date_valid_from').is_null()))
print(df_date_check.select(pl.col('date_valid_to').null_count()))

In [ ]:
print(df_org['registry'])
print(df_org.filter(pl.col('registry').str.contains(r'^[^a-zA-Z]*W/')))
print(df_org.filter(pl.col('registry').str.contains(r'^[^a-zA-Z]*D/')))
